In [ ]:
import numpy as np
import polars as pl

from embedding import Word2VecParser, Word2Vec

import matplotlib.pyplot as plt

RAW_DATA = "data/corpora.txt"

In [ ]:
texts = pl.scan_ndjson('hf://datasets/sentence-transformers/reddit-title-body/**/*.jsonl.gz').head(50_000).collect()
with open(RAW_DATA, mode='w+') as file:
    file.writelines(texts['body'])
(
    texts.select(
        pl.col('body').str.len_chars().alias("len_chars"),
        pl.col('body').str.split(by=' ').list.len().alias("len_words")
    ).describe()
)

## Word2Vec Implementation

Here you can find a demonstration of training a word2vec skip-gram version and some analysis of the results. Wikipedia articles are used to train embeddings, evaluation is performed using spearman's correlation on the WordSim353 dataset.

In [ ]:
# helper functions

def viz_losses(train_losses, val_losses):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 4))
    plt.plot(epochs, train_losses, label="train loss", lw=2, marker='o')
    plt.plot(epochs, val_losses, label="validation loss", lw=2, marker='o')
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Training and validation loss")
    plt.legend()
    plt.grid(True)
    plt.show()


def viz_norms(norms_dict: dict, title):
    plt.figure(figsize=(12, 4))
    for param_name in norms_dict.keys():
        plt.plot(range(1, len(norms_dict[param_name]) + 1), norms_dict[param_name], label=param_name)
    plt.xlabel("step")
    plt.ylabel("l2-norm")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def cosine_similarity(first_embedding, second_embedding):
    return first_embedding @ second_embedding / (np.linalg.norm(first_embedding) * np.linalg.norm(second_embedding))

In [ ]:
import pickle

# with open('data/vocab_default.pkl', 'wb') as file:
#     pickle.dump(vocab_default, file)

# with open('data/vocab_default.pkl', 'rb') as file:
#     vocab_default = pickle.load(file)

In [ ]:
filename = 'data/word_contexts_default.csv'
vocab = Word2VecParser(RAW_DATA).parse_positive_negative_samples(filename, k=5, window_size=4)
print(f'Vocab size: {len(vocab)}')

In [ ]:
data = pl.scan_csv(filename, has_header=False, with_column_names=lambda _: ['center', 'context', 'label'])
model = Word2Vec.from_scratch(dataset=data, vocab=vocab, embedding_size=100, batch_size=100_000)
model.train(epoches=10)
viz_losses(model.train_losses_, model.val_losses_)
model.save_pretrained('data/word2vec/baseline')

In [ ]:
filename = 'data/word_contexts_smaller_window.csv'
vocab = Word2VecParser(RAW_DATA).parse_positive_negative_samples(filename, window_size=3, k=2)
print(f'Vocab size: {len(vocab)}')

In [ ]:
data = pl.scan_csv(filename, has_header=False, with_column_names=lambda _: ['center', 'context', 'label'])
model = Word2Vec.from_scratch(dataset=data, vocab=vocab, embedding_size=100, batch_size=100_000)
model.train(epoches=10)

viz_losses(model.train_losses_, model.val_losses_)
model.save_pretrained('data/word2vec/smaller_window')

In [ ]:
filename = 'data/word_contexts_bigger_window.csv'
vocab = Word2VecParser(RAW_DATA).parse_positive_negative_samples(filename, window_size=7, k=6)
print(f'Vocab size: {len(vocab)}')

In [ ]:
data = pl.scan_csv(filename, has_header=False, with_column_names=lambda _: ['center', 'context', 'label'])
model = Word2Vec.from_scratch(dataset=data, vocab=vocab, embedding_size=100, batch_size=100_000)
model.train(epoches=10)
viz_losses(model.train_losses_, model.val_losses_)
model.save_pretrained('data/word2vec/bigger_window')

In [ ]:
filename = 'data/word_contexts_bigger_x2_window.csv'
vocab = Word2VecParser(RAW_DATA).parse_positive_negative_samples(filename, window_size=9, k=8)
print(f'Vocab size: {len(vocab)}')

In [ ]:
data = pl.scan_csv(filename, has_header=False, with_column_names=lambda _: ['center', 'context', 'label'])
model = Word2Vec.from_scratch(dataset=data, vocab=vocab, embedding_size=100, batch_size=100_000)
model.train(epoches=10)
viz_losses(model.train_losses_, model.val_losses_)
model.save_pretrained('data/word2vec/bigger_x2_window')